# 02 — Verify training data
Dataset generation already happened in 00, so the baseline uses exactly the split
that training will use. This notebook inspects complete three-image SFT examples.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif (ROOT / "manga_notebook_codebase").is_dir():
    ROOT = ROOT / "manga_notebook_codebase"
assert (ROOT / "TaskData/sequences.json").is_file(), "Run from the project or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
from data_utils import load_jsonl, save_jsonl
from dataset_utils import prepare_datasets, read_dataset, image_paths, save_json
TASK_ROOT = ROOT / "TaskData"
OUTPUTS = ROOT / "outputs"
split = prepare_datasets(ROOT)  # reuses and validates the persisted split
MAX_PIXELS = 256 * 28 * 28  # shared 8 GB GPU budget for inference and training
MAX_NEW_TOKENS = 4096  # inspect raw output if a long sequence reaches the limit
print(ROOT, {k: len(split[k]) for k in ("train", "val")})

In [ ]:
train_rows, val_rows = read_dataset(ROOT, "train"), read_dataset(ROOT, "val")
assert {x["sequence_id"] for x in train_rows} == set(split["train"])
assert {x["sequence_id"] for x in val_rows} == set(split["val"])
for rows in (train_rows, val_rows):
    for row in rows:
        assert len(row["images"]) == len(json.loads(row["target"])["pages"]) == 3
        assert all(p.is_file() for p in image_paths(ROOT, row))
sample = train_rows[0]
print("Training sequences:", len(train_rows), "Validation:", len(val_rows))
print("Images:", sample["images"])
print("Prompt:", sample["prompt"])
print("Assistant target:", sample["target"])

No page-level splitting, token truncation, or synthetic examples are used.
Loss is applied to the assistant JSON and end-of-turn tokens only. Character names
are arbitrary and evaluated through a sequence-wide permutation, not literal names.
UNKNOWN reference speakers are unresolved and excluded from speaker diagnostics.